```
CONFIGURATION MINIMALE RECOMMANDEE
Python  3.13.x ou 3.12.x
PySpark 4.x
pandas  2.2.3
Java    21.x ou 25.x
```


In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pathlib import Path
import os
from dotenv import load_dotenv

In [2]:
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 14g pyspark-shell"
load_dotenv()

True

In [3]:
project_path = Path.cwd()
flight_folder_path = project_path / "Data" / "Bronze" / "Stats_US_Flights_2019_2026"
passenger_folder_path = project_path / "Data" / "Bronze"/ "T_T100D_MARKET_US_CARRIER_ONLY"
runways_folder_path = project_path / "Data" / "Bronze" / "Runways"

flight_silver_path = project_path / "Data" / "Silver" / "Flights"
passengers_silver_path = project_path / "Data" / "Silver" / "Passengers"

JDBC_DRIVER_PATH = (project_path / "Drivers" / "mssql-jdbc-13.6.0.jre11.jar")

SQLSERVER_NATIVE_PATH = project_path / "Drivers"

os.environ["PATH"] = (str(SQLSERVER_NATIVE_PATH) + os.pathsep + os.environ["PATH"])

In [4]:

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("DomesticFlightDelays")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.driver.extraClassPath", str(JDBC_DRIVER_PATH))
    .getOrCreate()
)


In [5]:
flight_files = [
    file.as_posix()
    for file in flight_folder_path.glob("*.csv")
]

passenger_files = [
    file.as_posix()
    for file in passenger_folder_path.glob("*.csv")
]

runways_files = [
    file.as_posix()
    for file in runways_folder_path.glob("*.csv")
]


In [6]:
COLUMNS_TO_KEEP_FLIGHTS = [
    "Year",
    "Quarter",
    "Month",
    "DayofMonth",
    "DayOfWeek",
    "FlightDate",
    "Reporting_Airline",
    "DOT_ID_Reporting_Airline",
    "OriginAirportID",
    "Origin",
    "OriginCityName",
    "OriginState",
    "OriginStateName",
    "DestAirportID",
    "Dest",
    "DestCityName",
    "DestState",
    "DestStateName",
    "CRSDepTime",
    "DepTime",
    "DepDelay",
    "TaxiOut",
    "WheelsOff",
    "WheelsOn",
    "TaxiIn",
    "CRSArrTime",
    "ArrTime",
    "ArrDelay",
    "Cancelled",
    "CancellationCode",
    "CRSElapsedTime",
    "ActualElapsedTime",
    "AirTime",
    "Distance",
    "DistanceGroup",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay",
    "DivAirportLandings",
    "DivReachedDest",
    "DivActualElapsedTime",
    "DivArrDelay",
    "DivDistance"
]


FLIGHT_SCHEMA = StructType([

    StructField("Year", IntegerType(), True),
    StructField("Quarter", IntegerType(), True),
    StructField("Month", IntegerType(), True),
    StructField("DayofMonth", IntegerType(), True),
    StructField("DayOfWeek", IntegerType(), True),
    StructField("FlightDate", DateType(), True),

    StructField("Reporting_Airline", StringType(), True),
    StructField("DOT_ID_Reporting_Airline", IntegerType(), True),

    StructField("OriginAirportID", IntegerType(), True),
    StructField("Origin", StringType(), True),
    StructField("OriginCityName", StringType(), True),
    StructField("OriginState", StringType(), True),
    StructField("OriginStateName", StringType(), True),

    StructField("DestAirportID", IntegerType(), True),
    StructField("Dest", StringType(), True),
    StructField("DestCityName", StringType(), True),
    StructField("DestState", StringType(), True),
    StructField("DestStateName", StringType(), True),

    StructField("CRSDepTime", IntegerType(), True),
    StructField("DepTime", IntegerType(), True),
    StructField("DepDelay", DoubleType(), True),

    StructField("TaxiOut", DoubleType(), True),
    StructField("WheelsOff", IntegerType(), True),
    StructField("WheelsOn", IntegerType(), True),
    StructField("TaxiIn", DoubleType(), True),

    StructField("CRSArrTime", IntegerType(), True),
    StructField("ArrTime", IntegerType(), True),
    StructField("ArrDelay", DoubleType(), True),

    StructField("Cancelled", DoubleType(), True),
    StructField("CancellationCode", StringType(), True),

    StructField("CRSElapsedTime", DoubleType(), True),
    StructField("ActualElapsedTime", DoubleType(), True),
    StructField("AirTime", DoubleType(), True),

    StructField("Distance", DoubleType(), True),
    StructField("DistanceGroup", DoubleType(), True),

    StructField("CarrierDelay", DoubleType(), True),
    StructField("WeatherDelay", DoubleType(), True),
    StructField("NASDelay", DoubleType(), True),
    StructField("SecurityDelay", DoubleType(), True),
    StructField("LateAircraftDelay", DoubleType(), True),

    StructField("DivAirportLandings", IntegerType(), True),
    StructField("DivReachedDest", DoubleType(), True),
    StructField("DivActualElapsedTime", DoubleType(), True),
    StructField("DivArrDelay", DoubleType(), True),
    StructField("DivDistance", DoubleType(), True)
])

In [7]:
FLIGHT_COLUMN_MAPPING = {
    "Year": "YEAR",
    "Quarter": "QUARTER",
    "Month": "MONTH",
    "DayofMonth": "DAY_OF_MONTH",
    "DayOfWeek": "DAY_OF_WEEK",
    "FlightDate": "FLIGHT_DATE",

    "Reporting_Airline": "REPORTING_AIRLINE",
    "DOT_ID_Reporting_Airline": "DOT_ID_REPORTING_AIRLINE",

    "OriginAirportID": "ORIGIN_AIRPORT_ID",
    "Origin": "ORIGIN",
    "OriginCityName": "ORIGIN_CITY_NAME",
    "OriginState": "ORIGIN_STATE",
    "OriginStateName": "ORIGIN_STATE_NAME",

    "DestAirportID": "DEST_AIRPORT_ID",
    "Dest": "DEST",
    "DestCityName": "DEST_CITY_NAME",
    "DestState": "DEST_STATE",
    "DestStateName": "DEST_STATE_NAME",

    "CRSDepTime": "CRS_DEP_TIME",
    "DepTime": "DEP_TIME",
    "DepDelay": "DEP_DELAY",

    "TaxiOut": "TAXI_OUT",
    "WheelsOff": "WHEEL_OFF",
    "WheelsOn": "WHEEL_ON",
    "TaxiIn": "TAXI_IN",

    "CRSArrTime": "CRS_ARR_TIME",
    "ArrTime": "ARR_TIME",
    "ArrDelay": "ARR_DELAY",

    "Cancelled": "CANCELLED",
    "CancellationCode": "CANCELLATION_CODE",

    "CRSElapsedTime": "CRS_ELAPSED_TIME",
    "ActualElapsedTime": "ACTUAL_ELAPSED_TIME",
    "AirTime": "AIR_TIME",

    "Distance": "DISTANCE",
    "DistanceGroup": "DISTANCE_GROUP",

    "CarrierDelay": "CARRIER_DELAY",
    "WeatherDelay": "WEATHER_DELAY",
    "NASDelay": "NAS_DELAY",
    "SecurityDelay": "SECURITY_DELAY",
    "LateAircraftDelay": "LATE_AIRCRAFT_DELAY",

    "DivAirportLandings": "DIV_AIRPORT_LANDINGS",
    "DivReachedDest": "DIV_REACHED_DEST",
    "DivActualElapsedTime": "DIV_ACTUAL_ELAPSED_TIME",
    "DivArrDelay": "DIV_ARR_DELAY",
    "DivDistance": "DIV_DISTANCE"
}

In [8]:
df_flights = (spark.read.csv(
        flight_files,
        header=True,
        inferSchema=False
    )
    .select(COLUMNS_TO_KEEP_FLIGHTS)
)

In [9]:
df_flights = df_flights.select(* [F.col(field.name).cast(field.dataType).alias(FLIGHT_COLUMN_MAPPING[field.name])for field in FLIGHT_SCHEMA.fields])

In [10]:
COLUMNS_TO_KEEP_PASSENGERS = [
    "DEPARTURES_SCHEDULED",
    "DEPARTURES_PERFORMED",
    "PAYLOAD",
    "SEATS",
    "PASSENGERS",
    "FREIGHT",
    "MAIL",
    "DISTANCE",
    "AIR_TIME",
    "UNIQUE_CARRIER",
    "AIRLINE_ID",
    "CARRIER_NAME",
    "ORIGIN_AIRPORT_ID",
    "ORIGIN",
    "ORIGIN_CITY_NAME",
    "ORIGIN_STATE_ABR",
    "ORIGIN_STATE_NM",
    "DEST_AIRPORT_ID",
    "DEST",
    "DEST_CITY_NAME",
    "DEST_STATE_ABR",
    "DEST_STATE_NM",
    "YEAR",
    "QUARTER",
    "MONTH",
    "DISTANCE_GROUP"
]

PASSENGERS_SCHEMA = StructType([
    StructField("DEPARTURES_SCHEDULED", DoubleType(), True),
    StructField("DEPARTURES_PERFORMED", DoubleType(), True),
    StructField("PAYLOAD", DoubleType(), True),
    StructField("SEATS", DoubleType(), True),
    StructField("PASSENGERS", DoubleType(), True),
    StructField("FREIGHT", DoubleType(), True),
    StructField("MAIL", DoubleType(), True),
    StructField("DISTANCE", DoubleType(), True),
    StructField("AIR_TIME", DoubleType(), True),
    StructField("UNIQUE_CARRIER", StringType(), True),
    StructField("AIRLINE_ID", IntegerType(), True),
    StructField("CARRIER_NAME", StringType(), True),
    StructField("ORIGIN_AIRPORT_ID", IntegerType(), True),
    StructField("ORIGIN", StringType(), True),
    StructField("ORIGIN_CITY_NAME", StringType(), True),
    StructField("ORIGIN_STATE_ABR", StringType(), True),
    StructField("ORIGIN_STATE_NM", StringType(), True),
    StructField("DEST_AIRPORT_ID", IntegerType(), True),
    StructField("DEST", StringType(), True),
    StructField("DEST_CITY_NAME", StringType(), True),
    StructField("DEST_STATE_ABR", StringType(), True),
    StructField("DEST_STATE_NM", StringType(), True),
    StructField("YEAR", IntegerType(), True),
    StructField("QUARTER", IntegerType(), True),
    StructField("MONTH", IntegerType(), True),
    StructField("DISTANCE_GROUP", IntegerType(), True)
])

In [11]:
df_passengers = (spark.read.csv(
        passenger_files,
        header=True,
        inferSchema=False,
    ).select(COLUMNS_TO_KEEP_PASSENGERS)
)

In [12]:
df_passengers = df_passengers.select(*[F.col(field.name)
                                       .cast(field.dataType)
                                       .alias(field.name) for field in PASSENGERS_SCHEMA.fields])


In [13]:
COLUMNS_TO_KEEP_AIRPORTS = [
    "STATE_CODE",
    "ARPT_ID",
    "ARPT_NAME",
    "CITY",
    "COUNTRY_CODE",
    "LAT1_DECIMAL",
    "LONG1_DECIMAL"
]

In [14]:
df_runways = spark.read.csv(runways_files, inferSchema= True, header= True).select(COLUMNS_TO_KEEP_AIRPORTS)

In [15]:
df_runways = df_runways.drop_duplicates(["ARPT_ID"])
rename_map = {
    "LAT1_DECIMAL": "LATITUDE",
    "LONG1_DECIMAL": "LONGITUDE"
}
df_runways = df_runways.withColumnsRenamed(rename_map)

In [16]:
df_flights = df_flights.drop_duplicates()

In [17]:
first_condition = F.col("DEP_DELAY").isNull() | F.col("ARR_DELAY").isNull()
second_condition = F.col("CRS_ELAPSED_TIME").isNull() | F.col("ACTUAL_ELAPSED_TIME").isNull()
third_condition = F.col("CANCELLED") == 1.00

df_flights = df_flights.filter(third_condition | ~(first_condition & second_condition))

In [18]:
df_flights = df_flights.join(df_runways, (df_flights["ORIGIN"] == df_runways["ARPT_ID"]) & (df_flights["ORIGIN_STATE"] == df_runways["STATE_CODE"]),
                              "inner").select(df_flights["*"], df_runways["LATITUDE"], df_runways["LONGITUDE"])

In [19]:
df_passengers = df_passengers.filter(F.col("PASSENGERS") != 0.0).sort("PASSENGERS")
df_passengers = df_passengers.drop_duplicates()

In [20]:
df_flights_silver = spark.read.parquet(flight_silver_path.as_posix())
df_passengers_silver =  spark.read.parquet(passengers_silver_path.as_posix())

In [21]:
first_columns = ["YEAR", "MONTH", "QUARTER"]

other_columns_passengers = [
    col for col in df_passengers_silver.columns
    if col not in first_columns
]

other_columns_flights = [
    col for col in df_flights_silver.columns
    if col not in first_columns
]

df_passengers_silver = df_passengers_silver.select(*first_columns, *other_columns_passengers)

df_flights_silver = df_flights_silver.select(*first_columns, *other_columns_flights)


In [ ]:
df_flights = df_flights.repartition("YEAR", "MONTH")

(df_flights.write
    .mode("overwrite")
    .partitionBy("YEAR", "MONTH")
    .parquet(flight_silver_path.as_posix())
)

In [37]:
df_passengers = df_passengers.repartition("YEAR", "MONTH")

(df_passengers.write
    .mode("overwrite")
    .partitionBy("YEAR", "MONTH")
    .parquet(passengers_silver_path.as_posix())
)

In [22]:
sql_user = os.getenv("SQLSERVER_USER")
sql_password = os.getenv("SQLSERVER_PASSWORD")
sql_database = os.getenv("SQLSERVER_DATABASE")
sql_table = os.getenv("SQL_SERVER_TABLE")
sql_second_table = os.getenv("SQL_SERVER_SECOND_TABLE")
sql_third_table = os.getenv("SQL_SERVER_THIRD_TABLE")
sql_server_url = (
    "jdbc:sqlserver://ICT-210-05;"
    "databaseName=StagingSilver;"
    "integratedSecurity=true;"
    "encrypt=true;"
    "trustServerCertificate=true;"
)

In [28]:
(
    df_flights_silver.write
        .format("jdbc")
        .option("url", sql_server_url)
        .option("dbtable", sql_table)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .option("batchsize", "50000")
        .option("numPartitions", "4")
        .mode("append")
        .save()
)


In [23]:
(
    df_passengers_silver.write
        .format("jdbc")
        .option("url", sql_server_url)
        .option("dbtable", sql_second_table)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .option("batchsize", "100000")
        .option("numPartitions", "6")
        .mode("append")
        .save()
)

In [43]:
(
    df_runways.write
        .format("jdbc")
        .option("url", sql_server_url)
        .option("dbtable", sql_third_table)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .option("batchsize", "100000")
        .option("numPartitions", "6")
        .mode("append")
        .save()
)

In [12]:
df_weather_delay = df_flights_silver.filter((F.col("WEATHER_DELAY") != 0.0) & (F.col("WEATHER_DELAY").isNotNull()) & (F.col("CANCELLED") == 0.0))
df_weather_cancelled = df_flights_silver.filter((F.col("CANCELLED") == 1.00) & (F.col("CANCELLATION_CODE") == "B") )


In [15]:
filter_column = ["FLIGHT_DATE", "ORIGIN","ORIGIN_CITY_NAME", "CRS_DEP_TIME","DEP_TIME","WEATHER_DELAY","LATITUDE", "LONGITUDE"]

df_weather_delay = df_weather_delay.select(filter_column).orderBy("FLIGHT_DATE", "CRS_DEP_TIME")
df_weather_cancelled = df_weather_cancelled.select(filter_column).orderBy("FLIGHT_DATE", "CRS_DEP_TIME")

In [18]:
df_weather_delay.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .option("sep", ",") \
    .csv(r"C:\Users\stgadmin\Documents\GitHub\Data-Engineering-project\weather")


df_weather_cancelled.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .option("sep", ",") \
    .csv(r"C:\Users\stgadmin\Documents\GitHub\Data-Engineering-project\cancelled")